# Minimum-time positioning of a mass — value iteration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro860/double_integrator_minimum_time.ipynb)

A unit mass slides on a line, pushed by a force limited to $|u| \le 1$. The task is to bring it **to rest at the origin as fast as possible**. The state is $x = [p,\; v]$ (position and speed), the dynamics are the double integrator
$$\dot x = f(x, u) = [\,v,\; u\,],$$
and the cost counts time: $g(x, u) = 1$ everywhere except inside a small target zone around the origin, where it is $0$. The optimal cost-to-go $J^*(x)$ is then the **time-to-go**, and the optimal policy is known to switch the force between its two limits (bang-bang).

**Value iteration** solves the Bellman equation on a grid,
$$J^*(x) = \min_{u} \Big[\, g(x,u)\,\Delta t + J^*\big(x + f(x,u)\,\Delta t\big) \Big],$$
by sweeping the whole grid until $J$ stops changing. This notebook shows the time-to-go and the policy it produces, what $J$ means after a **fixed number of sweeps**, the closed loop, and a check of any cost-to-go against the Bellman equation.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox.

In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np

from minilink import DoubleIntegrator, DynamicProgrammingPlanner, PlanningProblem, TimeCost

## 1. Plant

The catalog `DoubleIntegrator` is the mass. The bounds on the state set the extent of the grid (the box $[-2, 2]\,\mathrm{m} \times [-2, 2]\,\mathrm{m/s}$); the bounds on the input are the force limits. `EPS` is the radius of the target zone, `INF` the price charged for leaving the box.

In [ ]:
EPS = 0.1  # radius of the target zone around x = 0 where the cost stops
INF = 10.0  # price of leaving the box, and cap for the plots
X_GRID = (201, 201)
U_GRID = (3,)  # u in {-1, 0, +1}
DT = 0.05
X0 = np.array([1.2, 0.0])  # start: 1.2 m away, at rest

plant = DoubleIntegrator()
plant.state.lower_bound[:] = [-2.0, -2.0]
plant.state.upper_bound[:] = [+2.0, +2.0]
plant.inputs["u"].lower_bound = np.array([-1.0])
plant.inputs["u"].upper_bound = np.array([+1.0])
plant.x0 = X0

## 2. Cost: time

$$g(x, u) = \begin{cases} 0 & \|x\| < \varepsilon \\ 1 & \text{otherwise} \end{cases}, \qquad h(x) = 0 .$$

With this running cost, $J = \int g\, dt$ is the time spent outside the target zone: the cost-to-go of the optimal policy is the minimum time to reach it. `TimeCost` is this cost; `PlanningProblem` packages the plant, the cost and the goal.

In [ ]:
cost = TimeCost.from_system(plant, eps=EPS)
problem = PlanningProblem(plant, x_goal=np.zeros(2), cost=cost)

## 3. Value iteration

The grid has $201 \times 201$ states and three force levels, with an Euler step of $\Delta t = 0.05\,\mathrm{s}$. There is no discount ($\alpha = 1$): the sum stays finite because the target zone is absorbing at zero cost. `solve()` sweeps until the largest change in $J$ falls below `tol`.

In [ ]:
planner = DynamicProgrammingPlanner(
    problem,
    x_grid=X_GRID,
    u_grid=U_GRID,
    dt=DT,
    alpha=1.0,
    tol=1e-3,
    max_iterations=1000,
    out_of_bound_cost=INF,
    verbose=True,
)
planner.solve()

planner.plot_cost2go(jmax=INF, show_3d=True)
planner.plot_policy()
print(f"J*(x0) = {planner.value_at(X0):.2f} s")

The time-to-go surface has a crease along a curve through the origin, and the policy switches sign across it: on one side the mass accelerates toward the target, on the other it brakes. States near the box edge cost `INF`: from there, the force limit cannot keep the mass inside the box.

## 4. A fixed number of sweeps

`solve_steps(N)` runs exactly $N$ backward sweeps, the finite-horizon recursion of exact dynamic programming with horizon $N\,\Delta t$ and terminal cost $h = 0$. After $N$ sweeps, $J$ is the least time spent outside the zone **within** that horizon: it equals the time-to-go for states that can reach the zone in time, and saturates at $N\,\Delta t$ for the others — the recursion has not yet looked further than its horizon.

In [ ]:
for n_sweeps in (25, 250):
    planner.solve_steps(n_sweeps)
    planner.plot_cost2go(jmax=INF, title=f"J after {n_sweeps} sweeps (horizon {n_sweeps * DT:.2f} s)")
    planner.plot_policy()

## 5. Closed loop

`get_controller()` interpolates the policy table into a state feedback $u = \pi^*(x)$. The loop is integrated with the same Euler step as the grid, from `X0`, and the trajectory is drawn over the policy map.

In [ ]:
planner.solve()  # back to the converged solution
controller = planner.get_controller()
loop = controller @ plant

traj = loop.compute_trajectory(tf=6.0, n_steps=int(6.0 / DT) + 1, solver="euler")
loop.plot_trajectory(traj)
planner.plot_policy(trajectory=traj)

The arrival time read off the simulation, against the time-to-go the table predicts for the same start.

In [ ]:
inside = np.linalg.norm(traj.x, axis=0) < EPS
t_arrival = traj.t[inside][0] if inside.any() else np.inf
print(f"J*(x0) = {planner.value_at(X0):.2f} s    simulated arrival: {t_arrival:.2f} s")

## 6. Checking a cost-to-go against the Bellman equation

A field $J$ on the grid solves the discretized Bellman equation when, at every node,
$$J(x) - \min_{u} \Big[\, g(x,u)\,\Delta t + J\big(x + f(x,u)\,\Delta t\big) \Big] = 0 .$$
The left-hand side is the **Bellman residual**. The helper below evaluates it for any node-indexed field, using the grid's own successor table and interpolation — the same objects the planner sweeps. On the converged $J^*$ the residual is below `tol` well inside the feasible set; it grows only against the cliff, where the price of leaving the box enters the minimum. Any other candidate — a time-to-go you derive by hand, evaluated with `np.array([J(x) for x in grid.states])` — can be checked the same way.

In [ ]:
grid = planner.grid


def bellman_residual(J):
    """J(x) - min_u [ g(x,u) dt + J(x + f(x,u) dt) ] at every grid node, for a field J on the grid."""
    x_next = grid.x_next  # (nodes, actions, n): Euler successors x + f(x,u) dt
    J_next = grid.interpolate(J, x_next.reshape(-1, grid.n)).reshape(x_next.shape[:2])
    g = np.array([cost.g(x, np.zeros(1)) for x in grid.states])  # 1 outside the zone, 0 inside
    Q = g[:, None] * grid.dt + J_next  # one Q-value per (state, action)
    Q[~grid.x_next_ok] = INF  # successors that leave the box
    return J - Q.min(axis=1)


residual = bellman_residual(planner.result.J)
interior = planner.result.J < INF - 4.0  # away from the cliff
feasible = planner.result.J < INF - 1.0
print(f"max |residual| in the interior:       {np.abs(residual[interior]).max():.4f}")
print(f"max |residual| on the feasible set:   {np.abs(residual[feasible]).max():.4f}")
grid.plot_value(np.abs(residual), vmax=DT, title="|Bellman residual| of $J^*$")

## 7. Things to try

1. **Target zone.** Increase `EPS` (0.2, 0.5) and rerun sections 3 and 5. How do the time-to-go and the switching curve change, and why does a larger zone change the policy far from the origin?
2. **Sweeps.** Rerun section 4 with 10, 50 and 100 sweeps. Read the plateau value of $J$ each time, and say which states have their true time-to-go.
3. **Resolution.** Halve the grid (`X_GRID = (101, 101)`) and compare $J^*(x_0)$ and the simulated arrival time. Which one moves, and why?
4. **The analytical solution.** This problem has a closed-form optimal policy — a switching curve in the $(p, v)$ plane — and a closed-form time-to-go. Write both as functions of $x$, evaluate the time-to-go on `grid.states`, and pass it to `bellman_residual` to check that it satisfies the Bellman equation as well as the numerical solution does.